## In this notebook:

#### We concatenate parcellated PET images into region x receptor matrix of densities.
Adapted from Hansen Receptors (https://github.com/netneurolab/hansen_receptors)
Paper: https://www.nature.com/articles/s41593-022-01186-3

### Import Packages

In [1]:
import numpy as np
import pandas as pd
from netneurotools import datasets, plotting
from matplotlib.colors import ListedColormap
from scipy.stats import zscore
from nilearn.datasets import fetch_atlas_schaefer_2018
from netneurotools.datasets import fetch_cammoun2012
import nibabel as nib

In [2]:
import surfer

### Set Variables

In [4]:
path = '/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/'
datapath = '/Users/pecsok/projects/Neuromaps/hansen_receptors/'
figpath = "/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/parcellated/PET_parcellated/desikan_killiany/"
#'/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/analyses/figures/'

outpath = "/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/parcellated/PET_parcellated/desikan_killiany/"
atlas = 'desikan'

if atlas == 'schaefer':
    schaefer = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17) # This is sklearn.utils._bunch.Bunch. Other altases in this format? *Ask Golia
    nnodes = len(schaefer['labels'])
elif atlas == 'cammoun':
    scale = 'cammoun2012_scale500' #'scale1000_17' #'Harvard_Oxford_sub' 
    folder = 'atl-Cammoun2012_res-500'
    cammoun = fetch_cammoun2012()
    nnodes=1015
elif atlas == 'Harvard_Oxford_sub':
    nnodes = 21
elif atlas == 'desikan':
    nnodes = 112
    scale = 'desikan_killiany'
    desikan = '~/Desktop/desikan-killiany_aparcaseg.nii.gz'
    #desikan = '/Users/pecsok/nilearn_data/desikan_killiany/desikan-killiany2006_rois_1mm.nii.gz'
    atlas = 'desikan'
    parcel_file_path = '/Users/pecsok/nilearn_data/desikan_killiany/desikanKillianyNodeNames.txt'
    labels = pd.read_csv(parcel_file_path, header=None, names=['ParcelName'])
    
    lannot = '/Users/pecsok/Desktop/lh.aparc.DKTatlas.annot'
    rannot = '/Users/pecsok/Desktop/rh.aparc.DKTatlas.annot'
    labels_lh, ctab_lh, names_lh = nib.freesurfer.read_annot(lannot)
    labels_rh, ctab_rh, names_rh = nib.freesurfer.read_annot(rannot)
    hemispheres = ['Left'] * len(names_lh) + ['Right'] * len(names_rh)
    names = names_lh + names_rh
    #print(names)
    labels2 = pd.DataFrame({'ParcelName': names, 'Hemisphere': hemispheres})
    print(labels2)
    labels2['ParcelName'] = labels2['ParcelName'].apply(lambda x: x.decode('utf-8'))
    print(labels2)
    



                    ParcelName Hemisphere
0                   b'unknown'       Left
1                  b'bankssts'       Left
2   b'caudalanteriorcingulate'       Left
3       b'caudalmiddlefrontal'       Left
4            b'corpuscallosum'       Left
..                         ...        ...
67            b'supramarginal'      Right
68              b'frontalpole'      Right
69             b'temporalpole'      Right
70       b'transversetemporal'      Right
71                   b'insula'      Right

[72 rows x 2 columns]
                 ParcelName Hemisphere
0                   unknown       Left
1                  bankssts       Left
2   caudalanteriorcingulate       Left
3       caudalmiddlefrontal       Left
4            corpuscallosum       Left
..                      ...        ...
67            supramarginal      Right
68              frontalpole      Right
69             temporalpole      Right
70       transversetemporal      Right
71                   insula      Right

[72 

In [5]:
pd.set_option('display.max_rows', None)
#print(labels)

cortical = pd.read_csv("~/Desktop/cortical_significance_analysis.csv", sep=',')
subcortical = pd.read_csv("~/Desktop/subcortical_significance_analysis.csv", sep=',')


### Choose what to analyse

### Make Receptor Matrices

In [9]:
print(scale) #pecsok_pfns/neuromaps/data/parcellated/PET_parcellated/"+scale+"

desikan_killiany


In [6]:
#print(receptors_csv)
pd.set_option('display.max_rows', None)

In [65]:
#print(labels)

In [6]:
# Create annoying loop to reframe df to make equivalent of receptor_df

voldf = pd.DataFrame(index=labels['ParcelName'], columns=['Hemisphere','Volume_change','Pvalue']).fillna(0)

#print(voldf)
for i, roi in enumerate(cortical['ROI']):
    parts = roi.split('_')
    hemi = parts[0]
    string = parts[1]
    for j in range(len(voldf)):
        region = str(voldf.index[j])
        
        # Check if the extracted string is in the index of voldf
        if string.lower() in region.lower() and hemi.lower() in region.lower():
            # Add the corresponding volume to voldf
            value = cortical.loc[i, 'Estimate']
            p = cortical.loc[i, 'P.Value']
            voldf.at[region, 'Volume_change'] = value  # Use 'at' for fast scalar access
            voldf.at[region, 'Hemisphere'] = hemi
            voldf.at[region, 'Pvalue'] = p

# Subcortical
for i, roi in enumerate(subcortical['ROI']):
    parts = roi.split('.')
    hemi = parts[0]
    string = parts[1]
    print(string)
    for j in range(len(voldf)):
        region = str(voldf.index[j])
        
        # Check if the extracted string is in the index of voldf
        if string.lower() in region.lower() and hemi.lower() in region.lower():
            # Add the corresponding volume to voldf
            value = subcortical.loc[i, 'Estimate']
            p = cortical.loc[i, 'P.Value']
            voldf.at[region, 'Volume_change'] = value  # Use 'at' for fast scalar access
            voldf.at[region, 'Hemisphere'] = hemi
            voldf.at[region, 'Pvalue'] = p
            
# Display the resulting DataFrame
display(voldf)


Cerebellum
Thalamus
Caudate
Putamen
Pallidum
Hippocampus
Amygdala
Accumbens
VentralDC
Cerebellum
Thalamus
Caudate
Putamen
Pallidum
Hippocampus
Amygdala
Accumbens
VentralDC
Posterior
Mid_Posterior
Central
Mid_Anterior
Anterior


/var/folders/kk/w6xmtt2d55xfbxvb6wcqhq580000gp/T/ipykernel_67104/1880600848.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  voldf = pd.DataFrame(index=labels['ParcelName'], columns=['Hemisphere','Volume_change','Pvalue']).fillna(0)
/var/folders/kk/w6xmtt2d55xfbxvb6wcqhq580000gp/T/ipykernel_67104/1880600848.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '627.2975405' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  voldf.at[region, 'Volume_change'] = value  # Use 'at' for fast scalar access
/var/folders/kk/w6xmtt2d55xfbxvb6wcqhq580000gp/T/ipykernel_67104/1880600848.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and

,Hemisphere,Volume_change,Pvalue
ParcelName,,,
Left-Cerebral-White-Matter,0,0.000000,0.000000e+00
Left-Lateral-Ventricle,0,0.000000,0.000000e+00
Left-Inf-Lat-Vent,0,0.000000,0.000000e+00
Left-Cerebellum-White-Matter,Left,6680.831629,8.480000e-10
Left-Cerebellum-Cortex,Left,6680.831629,8.480000e-10
Left-Thalamus-Proper,Left,1673.816025,2.390010e-04
Left-Caudate,Left,585.087394,5.790000e-12
Left-Putamen,Left,1209.638209,8.720000e-07
Left-Pallidum,Left,262.890897,1.720425e-01


In [7]:
#Trim Voldf and only keep values that are in annot file
#


#print(voldf)

# Make new column for values
labels2['Volume_change'] = np.nan

print(labels2['ParcelName'])

# Loop through voldf and add correct data to correct row
for i, roi in enumerate(voldf.index):
    # Extract region string
    parts = roi.split('-')
    hemi = parts[0]
    if len(parts) > 1:
        string = parts[1]
    else:
        continue
    for j, surf_roi in enumerate(labels2['ParcelName']):
        # Extract Surface plot region string
        #print(surf_roi)
        if not isinstance(surf_roi, float):
            if string.lower() in surf_roi.lower():
                if pd.isna(labels2.loc[j,'Volume_change']):
                    #print(
                    value = voldf.loc[roi, 'Volume_change']
                    p = voldf.loc[roi, 'Pvalue']
                    labels2.at[j, 'Volume_change'] = value  # Use 'at' for fast scalar access
                    #labels2.at[i, 'Hemisphere'] = hemi
                    labels2.at[j, 'Pvalue'] = p
                    


0                      unknown
1                     bankssts
2      caudalanteriorcingulate
3          caudalmiddlefrontal
4               corpuscallosum
5                       cuneus
6                   entorhinal
7                     fusiform
8             inferiorparietal
9             inferiortemporal
10            isthmuscingulate
11            lateraloccipital
12        lateralorbitofrontal
13                     lingual
14         medialorbitofrontal
15              middletemporal
16             parahippocampal
17                 paracentral
18             parsopercularis
19               parsorbitalis
20            parstriangularis
21               pericalcarine
22                 postcentral
23          posteriorcingulate
24                  precentral
25                   precuneus
26    rostralanteriorcingulate
27        rostralmiddlefrontal
28             superiorfrontal
29            superiorparietal
30            superiortemporal
31               supramarginal
32      

"   \n\nfor i, roi in enumerate(cortical['ROI']):\n    parts = roi.split('_')\n    hemi = parts[0]\n    string = parts[1]\n    for j in range(len(voldf)):\n        region = str(voldf.index[j])\n        \n        # Check if the extracted string is in the index of voldf\n        if string.lower() in region.lower() and hemi.lower() in region.lower():\n            # Add the corresponding volume to voldf\n            value = cortical.loc[i, 'Estimate']\n            p = cortical.loc[i, 'P.Value']\n            voldf.at[region, 'Volume_change'] = value  # Use 'at' for fast scalar access\n            voldf.at[region, 'Hemisphere'] = hemi\n            voldf.at[region, 'Pvalue'] = p\n\n\n   \n            if string.lower() in region.lower() and hemi.lower() in region.lower():\n            # Add the corresponding volume to voldf\n            value = cortical.loc[i, 'Estimate']\n            p = cortical.loc[i, 'P.Value']\n            voldf.at[region, 'Volume_change'] = value  # Use 'at' for fast sca

In [21]:
print(labels2)

                   ParcelName Hemisphere  Volume_change        Pvalue
0                     unknown       Left            NaN           NaN
1                    bankssts       Left       0.000000  0.000000e+00
2     caudalanteriorcingulate       Left       0.000000  0.000000e+00
3         caudalmiddlefrontal       Left            NaN           NaN
4              corpuscallosum       Left            NaN           NaN
5                      cuneus       Left            NaN           NaN
6                  entorhinal       Left            NaN           NaN
7                    fusiform       Left            NaN           NaN
8            inferiorparietal       Left            NaN           NaN
9            inferiortemporal       Left            NaN           NaN
10           isthmuscingulate       Left            NaN           NaN
11           lateraloccipital       Left            NaN           NaN
12       lateralorbitofrontal       Left            NaN           NaN
13                  

In [8]:
# VERSION 2 with aparc regions 
voldf = pd.DataFrame(index=labels['ParcelName'], columns=['Volume_change','Pvalue']).fillna(0)
hemispheres = ['Left'] * 36 + ['Right'] * 36
voldf['Hemisphere'] = hemispheres

#print(voldf)
#print(labels.shape)

#print(voldf)
for i, roi in enumerate(cortical['ROI']):
    parts = roi.split('_')
    hemi = parts[0]
    string = parts[1]
    for j in range(len(voldf)):
        #region = str(voldf.index[j])
        region = voldf.loc[j, 'Hemisphere']
        
        # Check if the extracted string is in the index of voldf
        if string.lower() in region.lower() and hemi.lower() in region.lower():
            # Add the corresponding volume to voldf
            value = cortical.loc[i, 'Estimate']
            p = cortical.loc[i, 'P.Value']
            voldf.at[region, 'Volume_change'] = value  # Use 'at' for fast scalar access
            voldf.at[region, 'Hemisphere'] = hemi
            voldf.at[region, 'Pvalue'] = p

# Subcortical
for i, roi in enumerate(subcortical['ROI']):
    parts = roi.split('.')
    hemi = parts[0]
    string = parts[1]
    print(string)
    for j in range(len(voldf)):
        region = str(voldf.index[j])
        
        # Check if the extracted string is in the index of voldf
        if string.lower() in region.lower() and hemi.lower() in region.lower():
            # Add the corresponding volume to voldf
            value = subcortical.loc[i, 'Estimate']
            p = cortical.loc[i, 'P.Value']
            voldf.at[region, 'Volume_change'] = value  # Use 'at' for fast scalar access
            voldf.at[region, 'Hemisphere'] = hemi
            voldf.at[region, 'Pvalue'] = p
            
# Display the resulting DataFrame
display(voldf)


/var/folders/kk/w6xmtt2d55xfbxvb6wcqhq580000gp/T/ipykernel_67104/2912069668.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  voldf = pd.DataFrame(index=labels['ParcelName'], columns=['Volume_change','Pvalue']).fillna(0)


ValueError: Length of values (72) does not match length of index (112)

In [13]:
# combine all the receptors (including repeats)
r = np.zeros([nnodes, len(receptors_csv)])
print(r.shape)
for i in range(len(receptors_csv)):
    r[:, i] = np.genfromtxt(receptors_csv[i], delimiter=',') 
    # Now we have the receptor data by parcel (row) for all 6 receptor maps (col)
    

#print(r[:,3])
print(r.shape)

receptor_names = np.array(["5HT1a", "5HT1b", "5HT2a", "5HT4", "5HT6", "5HTT", "A4B2",
                           "CB1", "D1", "D2", "DAT", "GABAa", "H3", "M1", "mGluR5",
                           "MOR", "NET", "NMDA", "VAChT"])
#np.save(path+'data/receptor_names_pet.npy', receptor_names)

# make final region x receptor matrix
receptor_data = np.zeros([nnodes, len(receptor_names)])

# Add 5HT1a Data
receptor_data[:, 0] = zscore(r[:, 0]) 
# Add "5HT2a", "5HT4", "5HT6", "5HTT", "A4B2", "CB1", "D1"
receptor_data[:, 2:9] = zscore(r[:, 3:10])
# Add "DAT", "GABAa", "H3", "M1"
receptor_data[:, 10:14] = zscore(r[:, 12:16])
# Add "MOR", "NET", "NMDA"
receptor_data[:, 15:18] = zscore(r[:, 19:22])

# weighted average of 5HT1B p943
receptor_data[:, 1] = (zscore(r[:, 1])*22 + zscore(r[:, 2])*65) / (22+65)
# weighted average of D2 flb457
receptor_data[:, 9] = (zscore(r[:, 10])*37 + zscore(r[:, 11])*55) / (37+55)
# weighted average of mGluR5 ABP688
receptor_data[:, 14] = (zscore(r[:, 16])*22 + zscore(r[:, 17])*28 + zscore(r[:, 18])*73) / (22+28+73)
# weighted average of VAChT FEOBV
receptor_data[:, 18] = (zscore(r[:, 22])*4 + zscore(r[:, 23])*5 + zscore(r[:, 24])*18) / (4+5+18)

print(receptors_csv[0]) #5HT1a

# Make df and add parcel labels
receptor_df = pd.DataFrame(receptor_data, columns=receptor_names,index=labels)

"""
#schaefer = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17)
#labels = schaefer.labels
#labels = [label.decode('utf-8') for label in labels]
cam = fetch_cammoun2012()
info = pd.read_csv(cammoun['info'], sep=',')
info = info[info['scale']== 'scale500']
hemisphere = info['hemisphere'].values
labels = info['label'].values + hemisphere
print(labels)
"""

receptor_df.index = labels
receptor_df.index.name = 'Parcel'

#print(receptor_df)
# Save df
receptor_df.to_csv(path+'results/receptor_data_'+scale+'.csv', index=True, header=True)



(112, 25)
(112, 25)
/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/parcellated/PET_parcellated/desikan_killiany/5HT1a_way_hc36_savli.csv


In [11]:
cammoun = fetch_cammoun2012()
cam = fetch_cammoun2012()
info = pd.read_csv(cammoun['info'], sep=',')
info = info[info['scale']== 'scale060']
hemisphere = info['hemisphere'].values
labels = info['label'].values + hemisphere
print(labels.shape)

(129,)


In [14]:
# Make vol change image
r = np.zeros([nnodes, 1])
print(r.shape)
r[:,0] = voldf['Volume_change']

vol_data = r

(112, 1)


In [84]:
"""
plot receptor data
"""

## NEWWWWWW
if scale == 'cammoun2012_scale500':
    cammoun = datasets.fetch_cammoun2012(version='fsaverage')
    cammoun = cammoun['scale500']
    print(cammoun)
    #annot = datasets.fetch_schaefer2018('fsaverage')['100Parcels7Networks']
    for k in range(len(receptor_names)):
        brain = plotting.plot_fsaverage(data=receptor_data[:, k],
                                        lhannot = cammoun[0],
                                        rhannot = cammoun[1],
                                        colormap='plasma',
                                        views=['lat', 'med'],
                                        data_kws={'representation': "wireframe"})
        brain.save_image(figpath + folder + '/surface_receptor_'+receptor_names[k]+'.png')




#receptor_data[:, 2:9] = r[:, 3:10]
print(receptor_data)
print(len(receptor_data))

print(r)


Surface(lh='/Users/pecsok/nnt-data/atl-cammoun2012/fsaverage/atl-Cammoun2012_space-fsaverage_res-500_hemi-L_deterministic.annot', rh='/Users/pecsok/nnt-data/atl-cammoun2012/fsaverage/atl-Cammoun2012_space-fsaverage_res-500_hemi-R_deterministic.annot')


IndexError: index 146 is out of bounds for axis 0 with size 114

In [ ]:
cammoun = fetch_cammoun2012 

In [33]:
#pip install mne


Using default location ~/mne_data for sample...
Creating /Users/pecsok/mne_data


  4%|█▌                                   | 69.8M/1.65G [00:03<00:59, 26.6MB/s]

KeyboardInterrupt: 

In [26]:
from neuromaps.datasets import fetch_annotation
from neuromaps import transforms
import nibabel as nib

"""
labels_lh, ctab_lh, names_rh = nib.freesurfer.read_annot(rannot)
print(labels_lh.shape)
print(len(names_rh))

rfake = np.random.random(36)
#vol_data[:, 0],
"""
fake = np.random.rand(72)

lannot = '/Users/pecsok/Desktop/lh.aparc.DKTatlas.annot'
rannot = '/Users/pecsok/Desktop/rh.aparc.DKTatlas.annot'

if scale == 'desikan_killiany':    
    brain = plotting.plot_fsaverage(data=fake, 
                                    lhannot = lannot,
                                    rhannot = rannot,
                                    colormap='plasma',
                                    views=['lat', 'med'],
                                    data_kws={'representation': "wireframe"})
    #brain.save_image(figpath + folder + '/surface_receptor_'+receptor_names[k]+'.png')


qt.qpa.window: <QNSWindow: 0x40e8d4530; contentView=<QNSView: 0x40e8d4130; QCocoaWindow(0x6000006c8420, window=QWidgetWindow(0x6000095aafa0, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These will stop working now that the window is recreated, and will result in exceptions when the observers are removed. Break in QCocoaWindow::recreateWindowIfNeeded to debug.
qt.qpa.window: <QNSWindow: 0x3ded27e80; contentView=<QNSView: 0x3ded27a80; QCocoaWindow(0x60000069b2e0, window=QWidgetWindow(0x600005bfa400, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These will stop working now that the window is recreated, and will result in exceptions when the observers are removed. Break in QCocoaWindow::recreateWindowIfNeeded to debug.
qt.qpa.window: <QNSWindow: 0x3ded93fb0; contentView=<QNSView: 0x3ded93bb0; QCocoaWindow(0x6000006d5080, window=QWidgetWindow(0x6000095e6220, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These wil

ValueError: array split does not result in an equal division

In [132]:
fsavg = transforms.mni152_to_fsaverage(desikan, '164k')
lhannot = fsavg[0]
ldata_array = lhannot.darrays[0].data
lunique_labels = np.unique(ldata_array)
llabels = [f'label_{int(label)}' for label in lunique_labels]
# Create a colormap with random RGB values for each unique label (or define your own)
lcolormap = np.zeros((len(lunique_labels) + 1, 4), dtype=np.int32)  # +1 for background
for i, label in enumerate(lunique_labels):
    colormap[i + 1] = [i * 40 % 255, i * 80 % 255, i * 120 % 255, 255]  # RGB + alpha

nib.freesurfer.write_annot(
    '/Users/pecsok/Desktop/atl-desikan_space-fsaverage_res-112_hemi-L_deterministic.annot',
    labels=ldata_array.astype(int),  # Label array from Gifti data
    ctab=lcolormap,                  # Color table
    names=llabels                    # List of region names
)  

# Right hemisphere
rhannot = fsavg[1]
rdata_array = rhannot.darrays[0].data
runique_labels = np.unique(rdata_array)
rlabels = [f'label_{int(label)}' for label in runique_labels]
# Create a colormap with random RGB values for each unique label (or define your own)
rcolormap = np.zeros((len(runique_labels) + 1, 4), dtype=np.int32)  # +1 for background
for i, label in enumerate(runique_labels):
    colormap[i + 1] = [i * 40 % 255, i * 80 % 255, i * 120 % 255, 255]  # RGB + alpha

nib.freesurfer.write_annot(
    '/Users/pecsok/Desktop/atl-desikan_space-fsaverage_res-112_hemi-R_deterministic.annot',
    labels=rdata_array.astype(int),  # Label array from Gifti data
    ctab=rcolormap,                  # Color table
    names=rlabels                    # List of region names
)
"""

In [ ]:
info = pd.read_csv(cammoun['info'], sep=',')
info = info[info['scale']== 'scale500']
hemisphere = info['hemisphere'].values
labels = info['label'].values + hemisphere


